In [1]:
# 1. Environment & Path Setup
import os
import sys

import duckdb

# Add the project root directory to sys.path so 'pipeline' is importable
sys.path.append(os.path.abspath('..'))

import numpy as np  # noqa: F401
import pandas as pd  # noqa: F401
from statsbombpy import sb  # noqa: F401

In [2]:
# Anchor path to project data folder
DB_PATH = '/home/kishor/Projects/ShotDist/shotdistributionpredictor/data/events.db'
# Open read-only connection
con = duckdb.connect(DB_PATH, read_only=True)

In [3]:
unique_pm_query = "SELECT DISTINCT player_id, match_id FROM events WHERE player_id IS NOT NULL;"

In [4]:
unique_pm_df = con.sql(unique_pm_query).df()

In [5]:
pm_sample = unique_pm_df.sample(1).reset_index(drop=True)

In [6]:
p = str(pm_sample.player_id[0])
n = str(pm_sample.match_id[0])

In [7]:
sample_events_query = f"SELECT * FROM events WHERE player_id = {p} AND match_id = {n};"

In [8]:
sample_events_df = con.sql(sample_events_query).df()

In [9]:
KEY_COLUMNS = ['player_id',
               'match_id',
               'team_id',
               'player',
               'team',
               'position',
               'type',
               'match_seconds',
               'teamsheet',
               'mandown',
               'location',
               'x',
               'y'
]

In [11]:
column_name_query = "DESCRIBE SELECT * FROM events;"
column_names = con.sql(column_name_query).df()
column_names = column_names['column_name']

In [12]:
pass_mask = column_names.str.startswith('pass', na=False)
PASS_COLUMNS = column_names[pass_mask].tolist()

In [13]:
shot_mask = column_names.str.startswith('shot', na=False)
SHOT_COLUMNS = column_names[shot_mask].tolist()

In [14]:
carry_mask = column_names.str.startswith('carry', na=False)
CARRY_COLUMNS = column_names[carry_mask].tolist()

In [15]:
ball_receipt_mask = column_names.str.startswith('ball_receipt', na=False)
RECEIPT_COLUMNS = column_names[ball_receipt_mask].tolist()

In [16]:
ball_recovery_mask = column_names.str.startswith('ball_recovery', na=False)
RECOVERY_COLUMNS = column_names[ball_recovery_mask].tolist()

In [17]:
interception_mask = column_names.str.startswith('interception', na=False)
INTERCEPTION_COLUMNS = column_names[interception_mask].tolist()

In [18]:
duel_mask = column_names.str.startswith('duel', na=False)
DUEL_COLUMNS = column_names[duel_mask].tolist()

In [19]:
ff_mask = column_names.str.startswith('50_50', na=False)
FF_COLUMNS = column_names[ff_mask].tolist()

In [ ]:
RECEPTION_TYPES = ['Ball Receipt*',
                   'Ball Recovery',
]

In [21]:
RELEASE_TYPES = ['Pass',
                 'Shot',
                 'Complete Carry']

In [22]:
ff_events_query = "SELECT * FROM events WHERE type='50/50'"
all_ff_events = con.sql(ff_events_query).df()

In [23]:
all_ff_events = all_ff_events.dropna(axis=1, how = 'all')

In [24]:
ff_value = all_ff_events["50_50"].str.extract(
    r"'name':\s*([^}]+)"
)

In [25]:
all_ff_events["50_50_outcome"] = ff_value

In [31]:
TYPE_OUTCOMES = [
    ('Duel','duel_outcome','Won'),
    ('Interception', 'interception_outcome','Won'),
    ('Pass', 'pass_type', 'Recovery'),
    ('Shot', 'shot_type', 'Recovery')
]

In [28]:
columns_str = ", ".join(KEY_COLUMNS)
# 2. Dynamically build the AND clauses for the dictionary filters
filter_clauses = []
filter_values = []

for col, val in TYPE_OUTCOMES.items():
    filter_clauses.append(f"AND {col} = ?")
    filter_values.append(val)

# Combine the dynamic lines into a single string block
dynamic_filters_str = "\n".join(filter_clauses)

query = f"""
    SELECT {columns_str}
    FROM events
    WHERE type = ANY(?)
    {dynamic_filters_str}
    ORDER BY match_seconds ASC;
"""

query_params = [RECEPTION_TYPES] + filter_values

In [30]:
print(query)
print(query_params)


    SELECT player_id, match_id, team_id, player, team, position, type, match_seconds, teamsheet, mandown, location, x, y
    FROM events
    WHERE type = ANY(?)
    AND duel_outcome = ?
AND interception_outcome = ?
AND pass_type = ?
AND shot_type = ?
    ORDER BY match_seconds ASC;

[['Ball Receipt*', 'Ball Recovery', 'Duel', '50/50', 'Interception', 'Pass', 'Shot'], 'Won', 'Won', 'Recovery', 'Recovery']


In [29]:
test_df = con.execute(query, query_params).df()

In [32]:
def _select_column(columns : list = []):
    """Take list of column names and get SELECT string."""
    if columns == []:
        col_str = '*'
    else:
        col_str = ", ".join(columns)
    return f"SELECT {col_str}"


def _type_param(type_param : tuple = ()):
    """Take 3-tuples of event type, column name and column value and get AND string."""
    if len(type_param) == 3:
        t = str(type_param[0])
        n = str(type_param[1])
        v = str(type_param[2])
        return f"(type = {t} AND {n} = {v})"
    else:
        return ""


def _type_noparam(type_noparam : str = ""):
    """Take type name and return a WHERE ready string."""
    if type_noparam == "":
        return type_noparam
    else:
        return f"(type = {type_noparam})"


def _col_param(col_param : tuple = ()):
    """Take 2-tuples of column name and column value and return a WHERE ready string."""
    if len(col_param) == 2:
        n = str(col_param[0])
        v = str(col_param[1])
        return f"({n} = {v})"
    else:
        return ""

def player_match_query(player_id : float, match_id : int, columns : list = [], type_params : list = [], type_noparams : list = [], col_params : list = []):
    INIT_FROM = "FROM events"
    init_select = _select_column(columns = columns)

    main_query = init_select + ' ' + INIT_FROM + f" WHERE (player_id = {player_id}) AND (match_id = {match_id}) AND "

    or_clauses = []
    for type_param in type_params:
        add_query = _type_param(type_param=type_param)
        if add_query == "":
            continue
        else:
            or_clauses.append(add_query)


    for type_noparam in type_noparams:
        add_query = _type_noparam(type_noparam=type_noparam)
        if add_query == "":
            continue
        else:
            or_clauses.append(add_query)


    for col_param in col_params:
        add_query = _col_param(col_param=col_param)
        if add_query == "":
            continue
        else:
            or_clauses.append(add_query)

    if or_clauses:
        combined_or = " OR ".join(or_clauses)
        main_query = f"{main_query} AND ({combined_or})"

    return main_query + ';'

In [ ]:
my_query = player_match_query()